In [1]:
import pandas as pd
import torch

from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)

from datasets import Dataset

In [2]:
# Import data and EDA
df = pd.read_csv("data/data")

print(df.head())

print("\nShape:",df.shape)

print("\nMissing Values:")
print(df.isnull().sum())


   ID                                               Joke
0   1  What did the bartender say to the jumper cable...
1   2  Don't you hate jokes about German sausage? The...
2   3  Two artists had an art contest... It ended in ...
3   4  Why did the chicken cross the playground? To g...
4   5   What gun do you use to hunt a moose? A moosecut!

Shape: (1622, 2)

Missing Values:
ID      0
Joke    0
dtype: int64


In [3]:
# Tokenizer and GPT-2 Model
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

# GPT-2 does not have a default padding token
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [4]:
# Extract Jokes and Tokenize
jokes = df["Joke"].astype(str).tolist()

formatted_jokes = []

for joke in jokes:
    words = joke.split()

    if len(words) > 3:
        prompt = " ".join(words[:3]).lower()
        formatted = f"Prompt: {prompt}\nJoke: {joke}"
        formatted_jokes.append(formatted)

dataset = Dataset.from_dict({"text": formatted_jokes})

# Tokenize
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=64
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)


Map:   0%|          | 0/1619 [00:00<?, ? examples/s]

In [5]:
# Split dataset and training
split_dataset = tokenized_dataset.train_test_split(test_size=0.1, seed=42)

tokenized_train = split_dataset["train"]
tokenized_val = split_dataset["test"]

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# Training
training_args = TrainingArguments(
    output_dir="./gpt2-jokes",
    num_train_epochs=5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    learning_rate=5e-5,
    weight_decay=0.01,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator
)

trainer.train()
trainer.save_model("./gpt2-joke-model")
tokenizer.save_pretrained("./gpt2-joke-model")

/Users/erickimchee/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,2.905328,2.681096
2,2.695562,2.680881
3,2.627030,2.686304
4,2.589560,2.701415
5,2.548271,2.712163


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/erickimchee/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/erickimchee/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./gpt2-joke-model/tokenizer_config.json', './gpt2-joke-model/tokenizer.json')

In [6]:
# Generation function
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

model.to(device)
model.eval()

def generate_joke(prompt_words, max_new_tokens=40):
    prompt = f"Prompt: {' '.join(prompt_words).lower()}\nJoke:"
    
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.9,
            repetition_penalty=1.2,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    
    full_text = tokenizer.decode(output[0], skip_special_tokens=True)

    if "Joke:" in full_text:
        return full_text.split("Joke:", 1)[1].strip()
    return full_text.strip()
    

In [7]:
generated = generate_joke(["two", "artists", "had"])
print(generated)
print("\n")

generated = generate_joke(["coffee", "cat", "laptop"])
print(generated)

Two Artists Had A Girlfriend, She Accidentally Unfunny With It... ...so she went to the opposite side. I told her i was only going with light and then he said no thanks


Coffee Cat I know when you hear it... It's called a caller in high office. So, the people at my gym have to catch this one before he getscha! ~Hey boy~
